In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.online_shop;
CREATE VOLUME IF NOT EXISTS workspace.online_shop.raw;

In [0]:
S    = "workspace.online_shop"
base = "/Volumes/workspace/online_shop/raw"

# fact_orders = first load only
spark.read.parquet(f"{base}/fact_orders_initial.parquet") \
     .write.format("delta").mode("overwrite").saveAsTable(f"{S}.fact_orders")

# dimensions
for n in ["dim_customer", "dim_date", "dim_category"]:
    spark.read.parquet(f"{base}/{n}.parquet") \
         .write.format("delta").mode("overwrite").saveAsTable(f"{S}.{n}")

spark.sql(f"SELECT COUNT(*) AS fact_rows_v0 FROM {S}.fact_orders").show()
spark.sql(f"SHOW TABLES IN {S}").show()

+------------+
|fact_rows_v0|
+------------+
|      197153|
+------------+

+-----------+------------+-----------+
|   database|   tableName|isTemporary|
+-----------+------------+-----------+
|online_shop|dim_category|      false|
|online_shop|dim_customer|      false|
|online_shop|    dim_date|      false|
|online_shop| fact_orders|      false|
+-----------+------------+-----------+



In [0]:
from delta.tables import DeltaTable

incr = spark.read.parquet(f"{base}/fact_orders_incr.parquet")
print("incoming rows:", incr.count())

tgt = DeltaTable.forName(spark, f"{S}.fact_orders")
(tgt.alias("t")
    .merge(incr.alias("s"), "t.order_id = s.order_id")
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute())

spark.sql(f"SELECT COUNT(*) AS fact_rows_now FROM {S}.fact_orders").show()

incoming rows: 2847
+-------------+
|fact_rows_now|
+-------------+
|       200000|
+-------------+



In [0]:
%sql
DESCRIBE HISTORY workspace.online_shop.fact_orders;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
2,2026-11-05T10:28:15.000Z,74905023066976,abdullah.fintech@gmail.com,MERGE,"Map(predicate -> [""(order_id#13580L = order_id#13543L)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> true, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}], clusteringOnWriteStatus -> null)",null,List(1240427231056027),93c61b49-0410-4059-adc7-fdd65e612af2,1105-102341-rr9txf8z-v2n,1,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 56093, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 0, executionTimeMs -> 3267, materializeSourceTimeMs -> 552, numTargetRowsInserted -> 2847, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1556, numTargetRowsUpdated -> 0, numOutputRows -> 2847, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 2847, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1082)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
1,2026-11-05T10:27:42.000Z,74905023066976,abdullah.fintech@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""delta.parquet.format.version"":""2.12.0"",""delta.enableRowTracking"":""true"",""delta.checkpointPolicy"":""v2"",""io.unitycatalog.tableId"":""9f75081e-519b-4122-a1a9-a9c3f309a5ff"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-cbbc7a5c-82af-41b7-906f-7a9d462d3a61"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-503ce1de-b393-41d3-8826-c4769456a677""}, statsOnLoad -> true, clusterByAuto -> true, clusteringOnWriteStatus -> Reason for skipping: Nonexistent clustering column, generally when replacing a Liquid table with a table without clustering columns)",null,List(1240427231056027),2592559f-01b4-49ac-87dc-57b8cf80cb84,1105-102341-rr9txf8z-v2n,0,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 3691356, numDeletionVectorsRemoved -> 0, numOutputRows -> 197153, numOutputBytes -> 3691356)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
0,2026-11-05T10:27:12.000Z,74905023066976,abdullah.fintech@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""delta.parquet.format.version"":""2.12.0"",""delta.enableRowTracking"":""true"",""delta.checkpointPolicy"":""v2"",""io.unitycatalog.tableId"":""9f75081e-519b-4122-a1a9-a9c3f309a5ff"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-cbbc7a5c-82af-41b7-906f-7a9d462d3a61"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-503ce1de-b393-41d3-8826-c4769456a677""}, statsOnLoad -> true, clusterByAuto -> true, clusteringOnWriteStatus -> Reason for skipping: Nonexistent clustering column, generally when replacing a Liquid table with a table without clustering columns)",null,List(1240427231056027),e50c3cf7-9fa0-43b9-8235-9a6629c75303,1105-102341-rr9txf8z-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 197153, numOutputBytes -> 3691356)",null,Databricks-Ru

In [0]:
%sql
SELECT version,
       timestamp,
       operation,
       operationMetrics['numOutputRows']        AS output_rows,
       operationMetrics['numTargetRowsInserted'] AS rows_inserted
FROM (DESCRIBE HISTORY workspace.online_shop.fact_orders)
ORDER BY version;

version,timestamp,operation,output_rows,rows_inserted
0,2026-11-05T10:27:12.000Z,CREATE OR REPLACE TABLE AS SELECT,197153,null
1,2026-11-05T10:27:42.000Z,CREATE OR REPLACE TABLE AS SELECT,197153,null
2,2026-11-05T10:28:15.000Z,MERGE,2847,2847


In [0]:
%sql
SELECT 'version 0' AS version, COUNT(*) AS n
FROM workspace.online_shop.fact_orders VERSION AS OF 0
UNION ALL
SELECT 'current', COUNT(*)
FROM workspace.online_shop.fact_orders;

version,n
version 0,197153
current,200000


**5. Processing with Spark**


In [0]:
from pyspark.sql import functions as F, Window

S = "workspace.online_shop"
spark.sql(f"USE {S}")

valid = (spark.table(f"{S}.fact_orders")
         .filter(~F.col("status").isin("cancelled", "returned"))
         .select("city", "category", "amount",
                 F.trunc(F.to_date("order_date"), "month").alias("month")))

# --- gold_monthly_sales: revenue, orders, avg order value by month and city ---
(valid.groupBy("month", "city")
      .agg(F.sum("amount").alias("revenue"),
           F.count("*").alias("order_count"),
           F.avg("amount").alias("avg_order_value"))
      .write.format("delta").mode("overwrite").option("overwriteSchema", "true")
      .saveAsTable(f"{S}.gold_monthly_sales"))

# --- gold_category_mix: revenue share of each category by month ---
w = Window.partitionBy("month")
(valid.groupBy("month", "category")
      .agg(F.sum("amount").alias("revenue"), F.count("*").alias("order_count"))
      .withColumn("revenue_share", F.col("revenue") / F.sum("revenue").over(w))
      .write.format("delta").mode("overwrite").option("overwriteSchema", "true")
      .saveAsTable(f"{S}.gold_category_mix"))

print("done: gold_monthly_sales, gold_category_mix")

done: gold_monthly_sales, gold_category_mix


In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {S}.gold_customer_features AS
WITH cc AS (
  SELECT customer_id, category,
         COUNT(*)                                  AS n,
         SUM(amount)                               AS spend,
         SUM(CASE WHEN source = 'app' THEN 1 ELSE 0 END) AS app_n,
         MAX(CAST(order_date AS DATE))             AS last_dt
  FROM {S}.fact_orders
  WHERE status NOT IN ('cancelled', 'returned')
  GROUP BY customer_id, category
),
ranked AS (
  SELECT *, ROW_NUMBER() OVER (PARTITION BY customer_id
                               ORDER BY n DESC, spend DESC) AS rn
  FROM cc
)
SELECT customer_id,
       SUM(spend)                                          AS total_spend,
       SUM(n)                                              AS n_orders,
       DATEDIFF(DATE'2026-10-24', MAX(last_dt))            AS days_since_last_order,
       SUM(app_n) / SUM(n)                                 AS app_share,
       MAX(CASE WHEN rn = 1 THEN category END)             AS favourite_category
FROM ranked
GROUP BY customer_id
""")
print("done: gold_customer_features")

done: gold_customer_features


In [0]:
spark.sql(f"""
SELECT 'gold_monthly_sales' AS tbl, COUNT(*) AS n FROM {S}.gold_monthly_sales
UNION ALL SELECT 'gold_category_mix', COUNT(*) FROM {S}.gold_category_mix
UNION ALL SELECT 'gold_customer_features', COUNT(*) FROM {S}.gold_customer_features
""").show()

spark.table(f"{S}.gold_customer_features").show(5)

+--------------------+----+
|                 tbl|   n|
+--------------------+----+
|  gold_monthly_sales| 220|
|   gold_category_mix| 132|
|gold_customer_fea...|2000|
+--------------------+----+

+-----------+------------------+--------+---------------------+--------------------+------------------+
|customer_id|       total_spend|n_orders|days_since_last_order|           app_share|favourite_category|
+-----------+------------------+--------+---------------------+--------------------+------------------+
|          1|502690.99000000005|      39|                    0|                 0.0|           Grocery|
|          2|1452276.4200000002|      50|                    8|                0.02|       Electronics|
|          3|        1979487.34|      59|                    0| 0.03389830508474576|           Grocery|
|          4|         514202.24|      32|                   34|             0.09375|           Grocery|
|          5|4106984.5700000008|     144|                    1|0.0208333333

In [0]:
%sql
SELECT city,
       SUM(CASE WHEN year(month) = 2025 AND month(month) <= 9 THEN revenue END) AS rev_2025,
       SUM(CASE WHEN year(month) = 2026 AND month(month) <= 9 THEN revenue END) AS rev_2026,
       ROUND(100 * (SUM(CASE WHEN year(month) = 2026 AND month(month) <= 9 THEN revenue END)
                  / SUM(CASE WHEN year(month) = 2025 AND month(month) <= 9 THEN revenue END) - 1), 1) AS growth_pct
FROM workspace.online_shop.gold_monthly_sales
GROUP BY city
ORDER BY growth_pct DESC;

city,rev_2025,rev_2026,growth_pct
Lahore,3.5207036944E8,3.7696215669999987E8,7.1
Multan,1.1341657239E8,1.2051898789000003E8,6.3
Karachi,4.205811030000001E8,4.4240653152999985E8,5.2
Faisalabad,1.4265695692000002E8,1.4305121755999997E8,0.3
Hyderabad,7.820637149E7,7.835747031000002E7,0.2
Rawalpindi,1.4715263157999998E8,1.4247260704000002E8,-3.2
Peshawar,1.1313469463000001E8,1.0831992210000002E8,-4.3
Islamabad,1.9861415700000006E8,1.8665816979000002E8,-6.0
Sialkot,8.475505824E7,7.629043375E7,-10.0
Quetta,4.656450659E7,4.0257254489999995E7,-13.5


**Lahore** grew fast ahead of Multan and Karachi.


In [0]:
%sql
SELECT city,
       CASE WHEN rain_mm > 1 THEN 'rainy' ELSE 'dry' END AS day_type,
       COUNT(*)                                           AS orders,
       COUNT(DISTINCT CAST(order_date AS DATE))           AS days,
       ROUND(COUNT(*) / COUNT(DISTINCT CAST(order_date AS DATE)), 1) AS avg_orders_per_day
FROM workspace.online_shop.fact_orders
WHERE city IN ('Lahore', 'Karachi', 'Islamabad') AND rain_mm IS NOT NULL
GROUP BY city, CASE WHEN rain_mm > 1 THEN 'rainy' ELSE 'dry' END
ORDER BY city, day_type;

city,day_type,orders,days,avg_orders_per_day
Islamabad,dry,15282,453,33.7
Islamabad,rainy,7096,209,34.0
Karachi,dry,44442,587,75.7
Karachi,rainy,5788,75,77.2
Lahore,dry,32757,507,64.6
Lahore,rainy,10092,155,65.1


**No**, rainy days have slightly more order per days which is very less effect.


In [0]:
%sql
SELECT category,
       ROUND(SUM(revenue) / SUM(order_count), 0) AS avg_order_value
FROM workspace.online_shop.gold_category_mix
GROUP BY category
ORDER BY avg_order_value DESC;

category,avg_order_value
Electronics,78494.0
Home,35105.0
Fashion,13437.0
Beauty,7067.0
Grocery,5335.0
Books,3227.0


**Electronics** with average order value of **78494**.


**6. Churn Model + ML Flow**


In [0]:
from pyspark.sql import functions as F

S   = "workspace.online_shop"
CUT, END = "2026-08-01", "2026-10-24"
cut = F.lit(CUT).cast("date")

fact = (spark.table(f"{S}.fact_orders")
        .select("customer_id", "category", "source",
                F.col("amount").cast("double").alias("amount"),
                F.to_date("order_date").alias("d")))

# FEATURES: only orders before the cutoff
features = (fact.filter(F.col("d") < cut)
    .groupBy("customer_id").agg(
        F.sum("amount").alias("total_spend"),
        F.count("*").alias("n_orders"),
        F.avg("amount").alias("avg_order_value"),
        F.datediff(cut, F.max("d")).alias("days_since_last_order"),
        F.sum(F.when(F.col("d") >= F.date_sub(cut, 90), 1).otherwise(0)).alias("orders_last_90d"),
        F.avg(F.when(F.col("source") == "app", 1.0).otherwise(0.0)).alias("app_share"),
        F.countDistinct("category").alias("n_categories")))

# LABEL: any order between cutoff and END means not churned
buyers = (fact.filter((F.col("d") >= cut) & (F.col("d") <= F.lit(END).cast("date")))
              .select("customer_id").distinct()
              .withColumn("bought_after", F.lit(1)))

churn = (features.join(buyers, "customer_id", "left")
         .withColumn("churned", F.when(F.col("bought_after").isNull(), 1).otherwise(0))
         .drop("bought_after"))

(churn.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
      .saveAsTable(f"{S}.churn_features"))

feat_version = (spark.sql(f"DESCRIBE HISTORY {S}.churn_features")
                .agg(F.max("version")).collect()[0][0])
print("features Delta version:", feat_version)
spark.table(f"{S}.churn_features").groupBy("churned").count().show()

features Delta version: 0
+-------+-----+
|churned|count|
+-------+-----+
|      0| 1998|
|      1|    2|
+-------+-----+



In [0]:
from pyspark.sql import functions as F

CUT, END = "2026-08-01", "2026-10-24"
cut = F.lit(CUT).cast("date")
late_start = F.date_sub(F.lit(END).cast("date"), 30)    # 2026-09-24

buyers30 = (fact.filter((F.col("d") >= late_start) & (F.col("d") <= F.lit(END).cast("date")))
                .select("customer_id").distinct()
                .withColumn("bought_late", F.lit(1)))

churn30 = (features.join(buyers30, "customer_id", "left")
           .withColumn("churned", F.when(F.col("bought_late").isNull(), 1).otherwise(0))
           .drop("bought_late"))

(churn30.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
        .saveAsTable(f"{S}.churn_features_30d"))
spark.table(f"{S}.churn_features_30d").groupBy("churned").count().show()

+-------+-----+
|churned|count|
+-------+-----+
|      0| 1852|
|      1|  148|
+-------+-----+



In [0]:
import mlflow, mlflow.sklearn, pandas as pd
from mlflow.models import infer_signature
from pyspark.sql import functions as F
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, recall_score, precision_score,
                             f1_score, roc_auc_score)

TABLE = f"{S}.churn_features_30d"
feat_version = (spark.sql(f"DESCRIBE HISTORY {TABLE}")
                .agg(F.max("version")).collect()[0][0])
print("features Delta version:", feat_version)

pdf   = spark.table(TABLE).toPandas()
FEATS = ["total_spend", "n_orders", "avg_order_value", "days_since_last_order",
         "orders_last_90d", "app_share", "n_categories"]
X, y, ids = pdf[FEATS], pdf["churned"], pdf["customer_id"]

X_tr, X_te, y_tr, y_te, id_tr, id_te = train_test_split(
    X, y, ids, test_size=0.25, random_state=42, stratify=y)
print("train:", len(X_tr), "| test:", len(X_te), "| churners in test:", int(y_te.sum()))

configs = {
  "logreg": (make_pipeline(StandardScaler(),
             LogisticRegression(class_weight="balanced", max_iter=1000)),
             {"model": "logistic_regression", "class_weight": "balanced"}),
  "rf_depth3": (RandomForestClassifier(n_estimators=100, max_depth=3, class_weight="balanced",
                random_state=42, n_jobs=-1),
             {"model": "random_forest", "n_estimators": 100, "max_depth": 3}),
  "rf_depth8": (RandomForestClassifier(n_estimators=100, max_depth=8, class_weight="balanced",
                random_state=42, n_jobs=-1),
             {"model": "random_forest", "n_estimators": 100, "max_depth": 8}),
}

run_ids = {}
for name, (model, params) in configs.items():
    with mlflow.start_run(run_name=name) as run:
        model.fit(X_tr, y_tr)
        pred  = model.predict(X_te)
        proba = model.predict_proba(X_te)[:, 1]

        mlflow.log_params(params)
        mlflow.log_param("features_table", TABLE)
        mlflow.log_param("features_delta_version", int(feat_version))
        mlflow.log_param("cutoff", "2026-08-01")
        mlflow.log_param("label", "no order 2026-09-24 to 2026-10-24")

        mlflow.log_metric("accuracy",  accuracy_score(y_te, pred))
        mlflow.log_metric("recall",    recall_score(y_te, pred, zero_division=0))
        mlflow.log_metric("precision", precision_score(y_te, pred, zero_division=0))
        mlflow.log_metric("f1",        f1_score(y_te, pred, zero_division=0))
        mlflow.log_metric("roc_auc",   roc_auc_score(y_te, proba))

        mlflow.sklearn.log_model(model, artifact_path="model",
                                 signature=infer_signature(X_te, pred))
        run_ids[name] = run.info.run_id
        print(name, "done")

print(run_ids)

features Delta version: 0
train: 1500 | test: 500 | churners in test: 37


/databricks/python/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/11/05 11:32:18 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
🔗 View Logged Model at: https://dbc-ff4a0360-46d3.cloud.databricks.com/ml/experiments/1240427231056027/models/m-ac69adf7745b485f9a1d7

logreg done


/databricks/python/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/11/05 11:32:26 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
🔗 View Logged Model at: https://dbc-ff4a0360-46d3.cloud.databricks.com/ml/experiments/1240427231056027/models/m-d26bdb1fe2c6407c973d4

rf_depth3 done


/databricks/python/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/11/05 11:32:32 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
🔗 View Logged Model at: https://dbc-ff4a0360-46d3.cloud.databricks.com/ml/experiments/1240427231056027/models/m-689b42c370864ed6b23ee

rf_depth8 done
{'logreg': 'f03db58f696343ab9c4856f614387e22', 'rf_depth3': 'f8e1039344544dbaa04af8fd11c54774', 'rf_depth8': '46592b9c444746e797130f47384f6bec'}


In [0]:
runs = mlflow.search_runs()
runs = runs[runs["run_id"].isin(run_ids.values())]
cols = ["tags.mlflow.runName", "run_id", "metrics.accuracy", "metrics.recall",
        "metrics.precision", "metrics.f1", "metrics.roc_auc"]
display(runs[cols].sort_values("metrics.recall", ascending=False))

tags.mlflow.runName,run_id,metrics.accuracy,metrics.recall,metrics.precision,metrics.f1,metrics.roc_auc
rf_depth3,f8e1039344544dbaa04af8fd11c54774,0.564,0.8378378378378378,0.12757201646090535,0.22142857142857142,0.778880392271321
logreg,f03db58f696343ab9c4856f614387e22,0.574,0.7837837837837838,0.12393162393162394,0.2140221402214022,0.7636448543575973
rf_depth8,46592b9c444746e797130f47384f6bec,0.838,0.43243243243243246,0.21052631578947367,0.2831858407079646,0.7571653727161286


In [0]:
best  = "rf_depth3"     
model = mlflow.sklearn.load_model(f"runs:/{run_ids[best]}/model")

s = X_te.head(5)
display(pd.DataFrame({
    "customer_id":       id_te.head(5).values,
    "churn_prediction":  model.predict(s),
    "churn_probability": model.predict_proba(s)[:, 1].round(3),
}))

customer_id,churn_prediction,churn_probability
688,0,0.024
1241,0,0.06
1818,0,0.087
1655,0,0.109
842,0,0.196
